# Module 307 — SQL Injection and LLM Output Handling

This offline worksheet uses Python's built-in SQLite library and a temporary, in-memory database. It demonstrates why executable SQL should not be assembled from untrusted text, and how a parameterized query keeps that text as a value. Nothing here connects to HTB or a remote database.

**Course connection:** an LLM-generated query is still untrusted output. A model refusal or prompt rule is not a database authorization control.

## 1. Set up harmless sample data

We create two tables: `users`, which the toy feature is meant to search, and `admin_data`, which the feature should not expose. The values are fictional. `sqlite3.connect(':memory:')` creates a database that exists only in this Python process.

In [ ]:
import sqlite3

connection = sqlite3.connect(':memory:')
cursor = connection.cursor()

cursor.execute('CREATE TABLE users (id INTEGER, username TEXT)')
cursor.execute("INSERT INTO users VALUES (1, 'alice')")
cursor.execute('CREATE TABLE admin_data (id INTEGER, note TEXT)')
cursor.execute("INSERT INTO admin_data VALUES (1, 'fictional training note')")

print(cursor.execute('SELECT id, username FROM users').fetchall())

## 2. See what unsafe string-building does

The ordinary request `alice` is interpolated directly into the SQL text. In the local demonstration only, the crafted value closes the quoted username, adds a `UNION SELECT` for the second table, and comments out the remaining quote. The database then interprets part of the supposed username as SQL syntax.

This is a toy reproduction of the trust-boundary mistake. Never test payloads against systems without authorization. The lesson's real defense is not to rely on the model or a text filter to keep SQL harmless.

In [ ]:
def unsafe_user_lookup(user_input):
    # The f-string inserts input directly into executable SQL text.
    query = f"SELECT id, username FROM users WHERE username = '{user_input}'"
    print('SQL sent to SQLite:', query)
    return cursor.execute(query).fetchall()

training_input = "alice' UNION SELECT id, note FROM admin_data -- "
unsafe_user_lookup(training_input)

## 3. Keep input as data with a parameterized query

The `?` is a placeholder. SQLite receives the SQL structure and the value separately. It does not reinterpret quote marks or SQL keywords inside `user_input` as query instructions. The injected-looking string should therefore match no username and return an empty list.

Parameterization protects values. If a user must choose a table name, column, or operation, those SQL identifiers cannot generally be supplied as ordinary value parameters; map the choice to a fixed allowlist in application code instead.

In [ ]:
def safe_user_lookup(user_input):
    # SQL structure is fixed; only the value is supplied separately.
    query = 'SELECT id, username FROM users WHERE username = ?'
    return cursor.execute(query, (user_input,)).fetchall()

print('Normal value:', safe_user_lookup('alice'))
print('SQL-looking value:', safe_user_lookup(training_input))

## 4. What to remember for the COAE exam

- **Confidentiality:** an unauthorized `SELECT` can expose rows or schema metadata.
- **Integrity:** `INSERT`, `UPDATE`, or `DELETE` can add, change, or remove data.
- Treat model output as untrusted at the SQL boundary. Prompt rules and model refusals are not authorization.
- Prefer fixed, parameterized queries, server-side authorization, and least-privilege database accounts.
- Distinguish prompt injection (influencing model behavior) from SQL injection (changing query structure); they may be chained together.

The first example is intentionally vulnerable and exists only in this in-memory toy database. Real applications should use the safe pattern and should not expose arbitrary SQL execution to a language model.